In [1]:
import os
import cv2
import random
import numpy as np
import joblib
from skimage.feature import hog
from tqdm import tqdm

In [2]:
# --- CONFIGURATION ---
# Pointing directly to your new single dataset folder
DATASET_PATH = r"Dataset\\asl_alphabet_train"

MAX_IMAGES_PER_CLASS = 1000 # Sweet spot for SVM processing time and accuracy
IMAGE_SIZE = (128, 128) # High-def resolution for U vs V distinction
OUTPUT_FILE = "final_hog_features.joblib"

In [3]:
def square_pad(crop):
    """Pad a grayscale crop to a square with black borders (no aspect distortion)."""
    h, w = crop.shape
    s = max(h, w)
    top = (s - h) // 2; bottom = s - h - top
    left = (s - w) // 2; right = s - w - left
    return cv2.copyMakeBorder(
            crop,
            top,
            bottom,
            left,
            right,
            cv2.BORDER_CONSTANT,
            value=0
        )


def segment_and_crop(img, image_size=(128, 128), padding=10):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if mask[mask.shape[0]//2, mask.shape[1]//2] == 0:
        mask = cv2.bitwise_not(mask)

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    c = max(contours, key=cv2.contourArea)
    if cv2.contourArea(c) < 1000:
        return None

    hx, hy, hw, hh = cv2.boundingRect(c)
    H, W = gray.shape
    hx = max(0, hx - padding); hy = max(0, hy - padding)
    hw = min(W - hx, hw + padding * 2); hh = min(H - hy, hh + padding * 2)

    crop = gray[hy:hy+hh, hx:hx+hw]
    if crop.size == 0:
        return None

    crop = square_pad(crop)               
    return cv2.resize(crop, image_size)

In [4]:
X_features = []
y_labels = []

# --- EXTRACTION LOOP ---
print(f"\nProcessing dataset: {DATASET_PATH}")

if not os.path.exists(DATASET_PATH):
    print(f"Error: Could not find the folder '{DATASET_PATH}'. Please check the path.")
else:
    # Get all class folders (A-Z, 0-9)
    class_folders = [f for f in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, f))]
    
    for class_name in tqdm(class_folders, desc="Extracting HOG Features"):
        class_folder = os.path.join(DATASET_PATH, class_name)
        all_images = os.listdir(class_folder)
        
        # Randomly shuffle and slice to ensure diverse training data
        if len(all_images) > MAX_IMAGES_PER_CLASS:
            selected_images = random.sample(all_images, MAX_IMAGES_PER_CLASS)
        else:
            selected_images = all_images
            
        for image_name in selected_images:
            img_path = os.path.join(class_folder, image_name)
            img = cv2.imread(img_path)
            
            if img is not None:
                # NEW
                resized_img = segment_and_crop(img, IMAGE_SIZE, padding=10)

                if resized_img is None:        # skip frames where no hand was segmented
                    continue
                
                features_hog = hog(
                    resized_img,
                    orientations=9,
                    pixels_per_cell=(16,16),
                    cells_per_block=(2,2), 
                    block_norm='L2-Hys', 
                    visualize=False
                )
                
                X_features.append(features_hog)
                y_labels.append(class_name.upper())
# --- PACKAGING AND SAVING ---
    print("\nExtraction complete! Converting to arrays...")
    X = np.array(X_features)
    y = np.array(y_labels)

    data_package = {
        "X": X,
        "y": y
    }

    print(f"Saving data to {OUTPUT_FILE}...")
    joblib.dump(data_package, OUTPUT_FILE, compress=9)

    print("-" * 40)
    print(" SUCCESS!")
    print(f" Total Images Processed: {len(y)}")
    print(f" Final Feature Shape: {X.shape}")
    print("-" * 40)


Processing dataset: Dataset\\asl_alphabet_train


Extracting HOG Features: 100%|██████████| 29/29 [01:12<00:00,  2.50s/it]



Extraction complete! Converting to arrays...
Saving data to final_hog_features.joblib...
----------------------------------------
 SUCCESS!
 Total Images Processed: 29000
 Final Feature Shape: (29000, 1764)
----------------------------------------
